# Claim Extractor Test Notebook

This notebook tests the `extract_claims` function from `claim_extractor.py` with two local model runs and one optional OpenAI remote run.

Models are loaded from `.env`, for example:
- `CLAIM_MODEL_1=Qwen/Qwen3.5-0.8B`
- `CLAIM_MODEL_2=Qwen/Qwen2.5-0.5B-Instruct`

Goals:
- Compare two local models
- Use direct prompt-based extraction only
- Quickly inspect claim count and claim content

In [17]:
import textwrap

from dotenv import dotenv_values

from src.claim_extraction.config import DOTENV_PATH
from src.claim_extraction.extractor import extract_claims

env = dotenv_values(DOTENV_PATH)

In [18]:
# Centralized config from .env (with defaults)
LOCAL_MODEL_1 = env.get('CLAIM_MODEL_1', 'Qwen/Qwen3.5-4B')
LOCAL_MODEL_2 = env.get('CLAIM_MODEL_2', 'Qwen/Qwen2.5-3B-Instruct')
REMOTE_MODEL = env.get('CLAIM_MODEL_REMOTE', 'gpt-4o-mini')
openai_api_key = env.get('OPENAI_API_KEY', '')

# Standard generation parameters used by all runs
TEMPERATURE = 0.1
MAX_NEW_TOKENS = 1536
VERBOSE = True

print('LOCAL_MODEL_1:', LOCAL_MODEL_1)
print('LOCAL_MODEL_2:', LOCAL_MODEL_2)
print('REMOTE_MODEL:', REMOTE_MODEL)
print('OPENAI_API_KEY:', '<set>' if openai_api_key else None)
print('TEMPERATURE:', TEMPERATURE)
print('MAX_NEW_TOKENS:', MAX_NEW_TOKENS)
print('VERBOSE:', VERBOSE)

LOCAL_MODEL_1: Qwen/Qwen3.5-4B
LOCAL_MODEL_2: Qwen/Qwen2.5-3B-Instruct
REMOTE_MODEL: gpt-4o-mini
OPENAI_API_KEY: <set>
TEMPERATURE: 0.1
MAX_NEW_TOKENS: 1536
VERBOSE: True


In [19]:
# Difficult but relatively short story
# story = textwrap.dedent('''
# In the valley of Meridia, a mouse named Tim wore a red brass-button coat and carried a map that his grandmother had drawn in 1984.
# Tim claimed the map showed three bridges over the river, but the mayor had announced last winter that only two bridges remained after flooding.
# At dawn, engineer Nora inspected the eastern bridge and told the council that the structure was safe for carts lighter than 500 kilograms.
# During the same meeting, historian Elias argued that the eastern bridge had already collapsed in 1999.
# Tim then said that if the eastern bridge was really safe, then every farmer from Oak District would deliver grain by sunset.
# By noon, two farmers from Oak District delivered grain, while four others reported blocked roads.
# In the afternoon, meteorologist Lina predicted heavy rain by evening, and she added that if rainfall exceeded 30 millimeters, then the western bridge would close automatically.
# The rain gauge later measured 34 millimeters.
# Nevertheless, the western bridge remained open until midnight according to the city logbook.
# Separately, archivist Omar wrote that there exists a lighthouse keeper named Rae who had never visited Meridia, even though a travel diary signed by Rae described a market visit in Meridia in 2022.
# At the festival, the announcer said the town had exactly 120 lanterns, while the inventory sheet listed 118 lanterns and two repaired frames.
# Finally, council minutes stated that if the budget was approved on Tuesday, then the school roof would be repaired before October, but the budget vote was postponed to Thursday.
# ''').strip()

# A longer story; this hits context-window limits more quickly.
story = textwrap.dedent("""
As goes Walmart, so goes the nation? Everyone from Apple CEO Tim Cook to the head of the NCAA slammed religious freedom laws being considered in several states this week, warning that they would open the door to discrimination against gay and lesbian customers.
But it was the opposition from Walmart, the ubiquitous retailer that dots the American landscape, that perhaps resonated most deeply, providing the latest evidence of growing opposition for gay rights in the heartland.
Walmart's staunch criticism of a religious freedom law in its home state of Arkansas came after the company said in February it would boost pay for about 500,000 workers well above the federal minimum wage.
Taken together, the company is emerging as a bellwether for shifting public opinion on hot-button political issues that divide conservatives and liberals.
And some prominent Republicans are urging the party to take notice.
Former Minnesota Gov. Tim Pawlenty, who famously called on the GOP to "be the party of Sam's Club, not just the country club," told CNN that Walmart's actions "foreshadow where the Republican Party will need to move."
"The Republican Party will have to better stand for" ideas on helping the middle class, said Pawlenty, the head of the Financial Services Roundtable, a Washington lobbying group for the finance industry.
The party's leaders must be "willing to put forward ideas that will help modest income workers, such as a reasonable increase in the minimum wage, and prohibit discrimination in things such as jobs, housing, public accommodation against gays and lesbians."
Walmart, which employs more than 50,000 people in Arkansas, emerged victorious on Wednesday.
Hours after the company's CEO, Doug McMillon, called on Republican Gov. Asa Hutchinson to veto the bill, the governor held a news conference and announced he would not sign the legislation unless its language was fixed.
Walmart's opposition to the religious freedom law once again puts the company at odds with many in the Republican Party, which the company's political action committee has tended to support.
In 2004, the Walmart PAC gave around $2 million to Republicans versus less than $500,000 to Democrats, according to data from the Center for Responsive Politics.
That gap has grown less pronounced in recent years.
In 2014, the PAC spent about $1.3 million to support Republicans and around $970,000 for Democrats.
It has been a gradual transformation for Walmart.
In 2011, the company bulked up its nondiscrimination policies by adding protections for gender identity.
Two years later, the company announced that it would start offering health insurance benefits to same-sex partners of employees starting in 2014.
Retail experts say Walmart's evolution on these issues over the years is partly a reflection of its diverse consumer base, as well as a recognition of the country's increasingly progressive views of gay equality (support for same-sex marriage is at a new high of 59%, according to a recent Wall Street Journal/NBC News poll).
"It's easy for someone like a Chick-fil-A to take a really polarizing position," said Dwight Hill, a partner at the retail consulting firm McMillanDoolittle.
"But in the world of the largest retailer in the world, that's very different."
Hill added: Same-sex marriage, "while divisive, it's becoming more common place here within the U.S., and the businesses by definition have to follow the trend of their customer."
The backlash over the religious freedom measures in Indiana and Arkansas this week is shining a bright light on the broader business community's overwhelming support for workplace policies that promote gay equality.
After Indiana Gov. Mike Pence, a Republican, signed his state's religious freedom bill into law, CEOs of companies big and small across the country threatened to pull out of the Hoosier state.
The resistance came from business leaders of all political persuasions, including Bill Oesterle, CEO of the business-rating website Angie's List and a one-time campaign manager for former Indiana Gov. Mitch Daniels.
Oesterle announced that his company would put plans on hold to expand its footprint in Indianapolis in light of the state's passage of the religious freedom act.
NASCAR, scheduled to hold a race in Indianapolis this summer, also spoke out against the Indiana law.
"What we're seeing over the past week is a tremendous amount of support from the business community who are standing up and are sending that equality is good for business and discrimination is bad for business," said Jason Rahlan, spokesman for the Human Rights Campaign.
The debate has reached presidential politics.
National Republicans are being forced to walk the fine line of protecting religious liberties and supporting nondiscrimination.
Likely GOP presidential candidate Jeb Bush initially backed Indiana's religious freedom law and Pence, but moderated his tone a few days later.
The former Florida governor said Wednesday that Indiana could have taken a "better" and "more consensus-oriented approach."
"By the end of the week, Indiana will be in the right place," Bush said, a reference to Pence's promise this week to fix his state's law in light of the widespread backlash.
Others in the GOP field are digging in.
Sen. Ted Cruz of Texas, the only officially declared Republican presidential candidate, said Wednesday that he had no interest in second-guessing Pence and lashed out at the business community for opposing the law.
"I think it is unfortunate that large companies today are listening to the extreme left wing agenda that is driven by an aggressive gay marriage agenda," Cruz said.
Meanwhile, former Secretary of State Hillary Clinton, who previously served on Walmart's board of directors, called on Hutchinson to veto the Arkansas bill, saying it would "permit unfair discrimination" against the LGBT community.
Jay Chesshir, CEO of the Little Rock Regional Chamber of Commerce in Arkansas, welcomed Hutchinson's pledge on Wednesday to seek changes to his state's bill.
He said businesses are not afraid to wade into a politically controversial debate to ensure inclusive workplace policies.
"When it comes to culture and quality of life, businesses are extremely interested in engaging in debate simply because it impacts its more precious resource -- and that's its people," Chesshir said.
"Therefore, when issues arise that have negative or positive impact on those things, then the business community will again speak and speak loudly."
""").strip()

print(f'Input length: {len(story)} chars')

Input length: 6510 chars


In [20]:
def run_case(label, **kwargs):
    print('\n' + '=' * 100)
    print(label)
    print('=' * 100)
    try:
        claims = extract_claims(text=story, **kwargs)
        print(f'Claim count: {len(claims)}')
        for i, c in enumerate(claims, 1):
            print(f'{i:02d}. {c}')
        return claims
    except Exception as exc:
        print(f'Run failed: {exc}')
        return []

In [21]:
# Direct extraction, local model 1
direct_local_model_1_claims = run_case(
    f'Direct | local | {LOCAL_MODEL_1}',
    model_name=LOCAL_MODEL_1,
    backend='local',
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    verbose=VERBOSE,
)


Direct | local | Qwen/Qwen3.5-4B
Starting claim extraction | backend=local | model=Qwen/Qwen3.5-4B | use_claimify=False


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Extracted 98 claims.
Claim count: 98
01. Walmart's performance is linked to the nation's performance.
02. Apple CEO Tim Cook slammed religious freedom laws being considered in several states this week.
03. The head of the NCAA slammed religious freedom laws being considered in several states this week.
04. Tim Cook and the head of the NCAA warned that the religious freedom laws would open the door to discrimination against gay and lesbian customers.
05. Walmart opposed religious freedom laws.
06. Walmart is a ubiquitous retailer.
07. Walmart dots the American landscape.
08. Walmart's opposition resonated deeply.
09. Walmart's opposition provided evidence of growing opposition for gay rights in the heartland.
10. Walmart criticized a religious freedom law in its home state of Arkansas.
11. Walmart's home state is Arkansas.
12. Walmart said in February it would boost pay for about 500,000 workers.
13. The pay boost for the workers would be well above the federal minimum wage.
14. Walmart

In [ ]:
# Direct extraction, local model 2
direct_local_model_2_claims = run_case(
    f'Direct | local | {LOCAL_MODEL_2}',
    model_name=LOCAL_MODEL_2,
    backend='local',
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    verbose=VERBOSE,
)


Direct | local | Qwen/Qwen2.5-3B-Instruct
Starting claim extraction | backend=local | model=Qwen/Qwen2.5-3B-Instruct | use_claimify=False


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

## OpenAI Remote Test

This test uses the remote OpenAI-compatible backend (`backend='remote'`).

Expected in `.env`:
- `OPENAI_API_KEY`
- `CLAIM_MODEL_REMOTE` (optional, fallback: `gpt-4o-mini`)

In [ ]:
# Direct extraction, OpenAI remote model
if not openai_api_key:
    print('OPENAI_API_KEY is missing in .env; skipping remote test.')
    direct_remote_openai_claims = []
else:
    direct_remote_openai_claims = run_case(
        f'Direct | remote | {REMOTE_MODEL}',
        model_name=REMOTE_MODEL,
        backend='remote',
        temperature=TEMPERATURE,
        max_new_tokens=MAX_NEW_TOKENS,
        verbose=VERBOSE,
    )


Direct | remote | gpt-4o-mini
Starting claim extraction | backend=remote | model=gpt-4o-mini | use_claimify=False
Extracted 70 claims.
Claim count: 70
01. Walmart is a ubiquitous retailer in the American landscape.
02. Tim Cook is the CEO of Apple.
03. The head of the NCAA slammed religious freedom laws being considered in several states this week.
04. Tim Cook warned that religious freedom laws would open the door to discrimination against gay and lesbian customers.
05. Walmart's criticism of a religious freedom law in Arkansas is staunch.
06. Walmart's criticism of the religious freedom law came after the company announced it would boost pay for about 500,000 workers well above the federal minimum wage.
07. Walmart is emerging as a bellwether for shifting public opinion on political issues that divide conservatives and liberals.
08. Some prominent Republicans are urging the Republican Party to take notice of Walmart's actions.
09. Tim Pawlenty is the former Governor of Minnesota.
10